## Mapping County Level Data 
Demo Code is from from Plotly Documentation 
https://plotly.com/python/mapbox-county-choropleth/

- Starting with plotly demo - moving to Project 1 saved data
- In Project 1 - FIPS to Lat Long conversion 
- In this notebook - FIPS to Shapes 


but... need to clean up county_fips from Projct 1 because some FIPS codes have leading zeros?


In [ ]:
import pandas as pd
from urllib.request import urlopen
import json
import matplotlib.pyplot as plt
import plotly.express as px

In [ ]:
try: 
    import contextily as ctx
except ImportError:
    %pip install contextily
    import contextily as ctx

try:
    import geopandas as gpd
except ImportError:
    %pip install geopandas
    import geopandas as gpd



## Start by importing a Geojson that has the shapes of the US Counties

In [ ]:

with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)

counties["features"][0]

Geometry has the Coordinates ( lat longs )  of the corners of the county

In [ ]:
# show the data for Alameda County, California
# just  save the json entry for alameda county 

alameda_county = next(item for item in counties["features"] if item["properties"]["NAME"] == "Alameda")
alameda_county

Save from Json to Dataframe


In [ ]:
alamdeda_df = pd.DataFrame(alameda_county["geometry"]["coordinates"])
alamdeda_df 

Reshape the data to lat long in columns and each row is a point


In [ ]:
alamdeda_df = pd.DataFrame(alameda_county["geometry"]["coordinates"][0])
alamdeda_df.columns = ["longitude", "latitude"]
alamdeda_df

Simple Matplotlib - plot the coordiates of alameda_df - in this dataset each column is a different lat long


In [ ]:
alamdeda_df.plot.scatter(x="longitude", y="latitude")


In [ ]:
# Convert DataFrame to GeoDataFrame
gdf = gpd.GeoDataFrame(alamdeda_df, geometry=gpd.points_from_xy(alamdeda_df["longitude"], alamdeda_df["latitude"]), crs="EPSG:4326")

# Convert to Web Mercator (for OSM basemap)
gdf = gdf.to_crs(epsg=3857)

In the following cell we can combine the matpotlib with a basemap using the package contextily 

In [ ]:
ax = gdf.plot(figsize=(10, 8), color='red', alpha=0.7, markersize=15)
ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)  # Add OSM basemap

ax.set_title("Alameda County Coordinates on OpenStreetMap")
plt.show()

## Back to the Plotly Demo  - Unemployment from 2016

In [ ]:
counties_df = pd.DataFrame(counties["features"])
counties_df.head()

Get Unemployment information - its was from plotly default dataset

In [ ]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/plotly/datasets/master/fips-unemp-16.csv",
                   dtype={"fips": str})
df.head()

In [ ]:

fig = px.choropleth_map(
    df, 
    geojson=counties, 
    locations='fips', 
    color='unemp',
    color_continuous_scale="Viridis",
    range_color=(0, 12),
    zoom=3, 
    center={"lat": 37.0902, "lon": -95.7129},
    opacity=0.5,
    labels={'unemp': 'Unemployment Rate'}
)

# Formatting
fig.update_layout(margin={"r":0, "t":0, "l":0, "b":0})
fig.show()

So Lets try this with the project 1 data!

In [ ]:
proj1_df = pd.read_csv("data/proj1.csv")
proj1_df

In [ ]:
proj1_df['county_fips'] = proj1_df['county_fips'].astype(str)


In [ ]:
#count the length of the county_fips code
proj1_df['county_fips'].apply(len).value_counts()

We need to make a function to add a zero to the front of the fips code only for certain states


In [ ]:
def add_zero(county_fips):
    if len(county_fips) == 4:
        return "0" + county_fips
    else:
        return county_fips

In [ ]:
proj1_df['county_fips'] = proj1_df['county_fips'].apply(add_zero)


In [ ]:
proj1_df['county_fips'].apply(len).value_counts()

In [ ]:
proj1_yr = proj1_df[proj1_df['year'] == 2019]
proj1_yr


In [ ]:
# redo the plotly map with proj1_yr
fig = px.choropleth_map(proj1_yr, geojson=counties, locations='county_fips', color='value',
                           color_continuous_scale="Viridis",
                           #range_color=(0, 12),
                           zoom=3, center = {"lat": 37.0902, "lon": -95.7129},
                           opacity=0.5,
                           labels={'rgdp growth':'pct change real gdp'},
                           hover_name='GeoName',  # This adds the County Name to the top of the tooltip 
                          )
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()

In [ ]:
fig = px.choropleth_map(
    proj1_yr, 
    geojson=counties, 
    locations='county_fips', 
    color='value',
    color_continuous_scale="Viridis",
    zoom=3, 
    center={"lat": 37.0902, "lon": -95.7129},
    opacity=0.5,
    labels={'rgdp growth': 'pct change real gdp'},
    hover_name='GeoName',
)

# Formatting
fig.update_layout(margin={"r":0, "t":0, "l":0, "b":0})
fig.show()

In [ ]:
# Subsample the data to just California using GeoFIPS
proj1_yr_ca = proj1_yr[proj1_yr['county_fips'].str.startswith("06")]
proj1_yr_ca

In [ ]:
# redo the plotly map with proj1_yr_ca
fig = px.choropleth_map(
    proj1_yr_ca, 
    geojson=counties, 
    locations='county_fips', 
    color='value', 
    color_continuous_scale="Viridis",
    zoom=5, 
    center={"lat": 37.0902, "lon": -120.7129},
    opacity=0.5,
    labels={'value': 'pct change real gdp'},
    hover_name='GeoName'
)

# Formatting
fig.update_layout(margin={"r":0, "t":0, "l":0, "b":0})
fig.show()